# Vision SFT · LaTeX OCR

Train Qwen3.5-2B to transcribe equation images using
[LaTeX OCR](https://huggingface.co/datasets/mlx-community/LaTeX_OCR).
Map raw `image` / `text` rows into conversations and learn from the LaTeX targets.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
from pathlib import Path

import mlx.core as mx
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.trainer import Trainer, VLMTrainingArgs, prepare_model_for_training
from mlx_vlm.trainer.datasets import CacheDataset
from mlx_vlm.trainer.vlm.sft.dataset import VisionDataset

## 1. Settings

Edit the model, data, and output path below. `iters` counts batches;
four accumulation steps make one optimizer update. Paths use the kernel's working directory.

Set sample limits to `None` to use the entire partition.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/LaTeX_OCR"
VALIDATION_FRACTION = 0.02
MAX_TRAIN_SAMPLES = 2048
MAX_EVAL_SAMPLES = 128
OCR_PROMPT = (
    "Transcribe the mathematical expression in this image into LaTeX. "
    "Return only the LaTeX expression, without explanations or Markdown fences."
)
OUTPUT_DIR = Path("adapters/vlm/sft_latex_ocr")
SEED = 42

# Adapters
TRAIN_TYPE = "lora"  # "lora", "dora", or "full"
CHECKPOINT_PATH = None
LORA_RANK = 8
LORA_ALPHA = 16

# Training
args = VLMTrainingArgs(
    iters=100,
    batch_size=1,
    learning_rate=1e-4,
    gradient_accumulation_steps=4,
    max_seq_length=2048,
    train_on_completions=True,
    cache_size=8,
    grad_checkpoint=True,
    compile=False,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()

## 2. Data

Hold out 2% of the training split for validation; keep the original test split for
final evaluation. `.map()` converts `text` into user/assistant messages and retains
the raw `image` column.


In [ ]:
raw_dataset = load_dataset(DATASET_ID)
required_columns = {"image", "text"}
if not required_columns.issubset(raw_dataset["train"].column_names):
    raise ValueError(
        f"Expected image/text columns, got {raw_dataset['train'].column_names}."
    )

partitions = raw_dataset["train"].train_test_split(
    test_size=VALIDATION_FRACTION,
    seed=SEED,
)


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


def latex_to_messages(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Every OCR row must contain a non-empty LaTeX text string.")
    return {
        "messages": [
            {"role": "user", "content": OCR_PROMPT},
            {"role": "assistant", "content": text},
        ],
    }


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES).map(
    latex_to_messages,
    input_columns=["text"],
    remove_columns=["text"],
    desc="Map training LaTeX into chat messages",
)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES).map(
    latex_to_messages,
    input_columns=["text"],
    remove_columns=["text"],
    desc="Map validation LaTeX into chat messages",
)

assert len(train_rows) >= args.batch_size, "Reduce batch_size or add training rows."
assert len(eval_rows) >= args.batch_size, "Validation needs a complete batch."
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)

## 3. Model

Preview the model's chat template with one image and the assistant target included.
Image decoding and token expansion happen during dataset preprocessing.


In [ ]:
model, processor = load(MODEL_ID)
sample = train_rows[0]
formatted_sample = apply_chat_template(
    processor,
    model.config,
    sample["messages"],
    num_images=1,
    add_generation_prompt=False,
)
print({"image_size": sample["image"].size})
print(formatted_sample[:2000])

### Prepare adapters

LoRA trains the language adapters while freezing the base. `TRAIN_TYPE` also accepts
`"dora"` or `"full"`; full training unfreezes all weights and needs more memory.
Set `CHECKPOINT_PATH` to saved weights to continue with a fresh optimizer.
See the [training guide](../../docs/training.md) for quantization and target-layer options.


In [ ]:
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    checkpoint_path=CHECKPOINT_PATH,
)

## 4. Train

VisionDataset applies the chat template, processes the raw image, and masks the prompt.
Only the LaTeX response contributes to the loss.


In [ ]:
train_vision_dataset = VisionDataset(
    train_rows,
    model.config,
    processor,
    config=args,
)
eval_vision_dataset = VisionDataset(
    eval_rows,
    model.config,
    processor,
    config=args,
)

trainer = Trainer(
    model,
    args,
    task="vlm",
    algorithm="sft",
    processing_class=processor,
    train_dataset=CacheDataset(
        train_vision_dataset, max_size=args.cache_size, process=False
    ),
    eval_dataset=CacheDataset(
        eval_vision_dataset, max_size=args.cache_size, process=False
    ),
    prepared=True,
    seed=SEED,
)

In [ ]:
metrics = trainer.train()

### Metrics

The table shows the last five training reports. All metrics remain in `metrics`
and `trainer.log_history`, including loss diagnostics, timing, and memory.
`trained_tokens` counts supervised targets; `total_vision_tokens` counts expanded
image/video input positions. Processing token/s includes forward, backward, and updates.


In [ ]:
report_history = pd.DataFrame(trainer.log_history)
columns = {
    "iteration": "step",
    "train_loss": "loss",
    "learning_rate": "lr",
    "trained_tokens": "trained tokens",
    "total_vision_tokens": "vision tokens",
    "processed_tokens_per_second": "processed tok/s",
    "peak_memory": "peak GB",
}
recent_reports = (
    report_history.dropna(subset=["train_loss"])
    .tail(5)
    .reindex(columns=columns)
    .rename(columns=columns)
    .set_index("step")
)
recent_reports.round({"loss": 4, "processed tok/s": 1, "peak GB": 2})

## 5. Evaluate

Check response loss on held-out images. To assess OCR quality, also compare generated
LaTeX with the targets on the original test split.


In [ ]:
assert mx.isfinite(mx.array(metrics["train_loss"])).item(), "Non-finite training loss."
validation = trainer.evaluate() if eval_rows is not None else {}
pd.DataFrame([validation]).filter(
    items=["val_loss", "val_token_accuracy", "val_perplexity", "val_num_vision_tokens"]
).round(4)

## 6. Save

Keep `weights.safetensors` and `adapter_config.json` together. The loop also saves
the final checkpoint automatically.


In [ ]:
trainer.save_adapter()
print(Path(args.adapter_file).resolve())

### Reload later

Load the original base model, then attach the saved weights:

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, checkpoint_path=args.adapter_file,
)
model.eval()
```

More options: [training guide](../../docs/training.md).
